# Vector Store vs. Vector Database

## Introduction

When building Retrieval-Augmented Generation (RAG) systems, semantic search engines, or AI applications utilizing embeddings, choosing between a **Vector Store** and a **Vector Database** is a critical architectural decision. Although both store and retrieve vector embeddings, they differ significantly in scalability, operational capabilities, and production readiness.

---

## Core Definitions

### What is a Vector Store?
A **Vector Store** is a lightweight library focused primarily on storing vector embeddings and performing similarity searches (e.g., Cosine Similarity, Euclidean Distance) to find nearest neighbors.

* Runs in-memory or via local files on a single machine.
* Best for research, local development, Proof of Concepts (POCs), and applications with under 1 million vectors.

### What is a Vector Database?
A **Vector Database** is a purpose-built, full-featured database system designed to manage vector data at massive scale, extending vector search with traditional database capabilities.

* Supports distributed deployment, replication, sharding, high availability, and advanced metadata filtering.
* Best for production AI applications, enterprise systems, multi-user environments, and datasets with billions of vectors.

---

## Comprehensive Comparison

| Feature | Vector Store | Vector Database |
|---------|--------------|-----------------|
| **Primary Purpose** | Store and search vectors locally | Manage vector data at enterprise scale |
| **Data Scale** | Small to medium (~1M vectors) | Massive (Billions+ vectors) |
| **Search Capabilities** | Basic similarity search | Similarity search + advanced metadata filtering |
| **CRUD Operations** | Basic or absent | Full Create, Read, Update, Delete support |
| **Query Speed** | Microseconds | Milliseconds (Highly optimized for scale) |
| **Deployment & Setup** | Local machine (Minutes to setup) | Distributed cloud infrastructure (Hours/Days to setup) |
| **Architecture** | Single machine, single point of failure | High availability, fault-tolerant, sharding, replication |
| **Multi-Tenancy** | Rare | Common |
| **Cost** | Free / Very Low | Medium to High |

---

## Architectural Concepts

* **CRUD Operations:** Unlike simple stores, vector databases support full CRUD functionality (Create, Read, Update, Delete), making them suitable for continuously changing production datasets where vectors must be updated or removed dynamically.
* **Replication:** Databases maintain multiple copies of data across different nodes. If one node fails, the data remains accessible from others, ensuring high availability and disaster recovery.
* **Sharding:** Databases can split massive datasets across multiple machines. Instead of storing 100 million vectors on one machine, sharding distributes them to enable faster queries, lower memory pressure, and efficient processing.

---

## Popular Tools Landscape

### Vector Stores (For Local/POC Use)
| Tool | Key Characteristics |
|------|---------------------|
| **FAISS (Meta)** | Extremely fast similarity search, works locally, highly popular for RAG. |
| **ChromaDB** | Beginner-friendly, easy local setup, tightly integrated with LangChain. |
| **Annoy (Spotify)** | Approximate nearest neighbor search, lightweight, memory efficient. |
| **ScaNN (Google)** | High-performance similarity search optimized for large datasets. |
| **NMSLIB** | Fast nearest-neighbor search with multiple indexing algorithms. |

### Vector Databases (For Production Use)
| Tool | Key Characteristics |
|------|---------------------|
| **Pinecone** | Fully managed cloud service, automatic scaling, enterprise-ready. |
| **Weaviate** | Open-source, hybrid search, GraphQL support, production-focused. |
| **Qdrant** | Open-source, high-performance, rich metadata filtering capabilities. |
| **Milvus** | Distributed architecture explicitly designed for massive scalability. |
| **DataStax Astra DB**| Cloud-native database built with enterprise-grade infrastructure. |

---

## Decision Guide & Real-World RAG

### When to Use a Vector Store
Choose a Vector Store for your **Beginner Stage**. When building an initial pipeline—where documents are converted to embeddings, stored locally, and passed to a Large Language Model (LLM)—tools like ChromaDB or FAISS are perfect for rapid experimentation, limited budgets, and personal learning.

### When to Use a Vector Database
Choose a Vector Database for your **Production Stage**. When scaling your application to handle millions of vectors, you need a robust pipeline. Platforms like Pinecone, Weaviate, or Qdrant are necessary to ensure high availability, support multiple simultaneous users, and filter large datasets efficiently before sending context to the LLM.

## Final Rule

> Use a **Vector Store** for experimentation, learning, and small-scale applications.
>
> Use a **Vector Database** for production systems, scalability, reliability, and enterprise-grade AI applications.

# Building a RAG System with LangChain and ChromaDB

## Introduction

Retrieval-Augmented Generation (RAG) is a powerful technique that combines the capabilities of large language models with external knowledge retrieval. This notebook will walk you through building a complete RAG system using:

- **LangChain:** A framework for developing applications powered by language models.
- **ChromaDB:** An open-source vector database for storing and retrieving embeddings.
- **OpenAI:** For embeddings and language model (you can substitute with other providers).

In [279]:
import os
from dotenv import load_dotenv
load_dotenv()

False

In [280]:
# !pip install langchain_classic langchain_core langchain_community langchain_google_genai

In [281]:
## langchain imports
from langchain_classic.text_splitter import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import TextLoader
from langchain_classic.schema import Document

## vectorstores
from langchain_community.vectorstores import Chroma

## utility imports
import numpy as np
from typing import List

In [282]:
# RAG Architecture Overview
print("""
RAG (Retrieval-Augmented Generation) Architecture:

1. Document Loading: Load documents from various sources
2. Document Splitting: Break documents into smaller chunks
3. Embedding Generation: Convert chunks into vector representations
4. Vector Storage: Store embeddings in ChromaDB
5. Query Processing: Convert user query to embedding
6. Similarity Search: Find relevant chunks from vector store
7. Context Augmentation: Combine retrieved chunks with query
8. Response Generation: LLM generates answer using context

Benefits of RAG:
- Reduces hallucinations
- Provides up-to-date information
- Allows citing sources
- Works with domain-specific knowledge
""")


RAG (Retrieval-Augmented Generation) Architecture:

1. Document Loading: Load documents from various sources
2. Document Splitting: Break documents into smaller chunks
3. Embedding Generation: Convert chunks into vector representations
4. Vector Storage: Store embeddings in ChromaDB
5. Query Processing: Convert user query to embedding
6. Similarity Search: Find relevant chunks from vector store
7. Context Augmentation: Combine retrieved chunks with query
8. Response Generation: LLM generates answer using context

Benefits of RAG:
- Reduces hallucinations
- Provides up-to-date information
- Allows citing sources
- Works with domain-specific knowledge



### 1. Sample Data

In [283]:
## create sample documents
sample_docs = [
    """
    Machine Learning Fundamentals

    Machine learning is a subset of artificial intelligence that enables systems to learn
    and improve from experience without being explicitly programmed. There are three main
    types of machine learning: supervised learning, unsupervised learning, and reinforcement
    learning. Supervised learning uses labeled data to train models, while unsupervised
    learning finds patterns in unlabeled data. Reinforcement learning learns through
    interaction with an environment using rewards and penalties.
    """,

    """
    Deep Learning and Neural Networks

    Deep learning is a subset of machine learning based on artificial neural networks.
    These networks are inspired by the human brain and consist of layers of interconnected
    nodes. Deep learning has revolutionized fields like computer vision, natural language
    processing, and speech recognition. Convolutional Neural Networks (CNNs) are particularly
    effective for image processing, while Recurrent Neural Networks (RNNs) and Transformers
    excel at sequential data processing.
    """,

    """
    Natural Language Processing (NLP)

    NLP is a field of AI that focuses on the interaction between computers and human language.
    Key tasks in NLP include text classification, named entity recognition, sentiment analysis,
    machine translation, and question answering. Modern NLP heavily relies on transformer
    architectures like BERT, GPT, and T5. These models use attention mechanisms to understand
    context and relationships between words in text.
    """
]

sample_docs

['\n    Machine Learning Fundamentals\n    \n    Machine learning is a subset of artificial intelligence that enables systems to learn \n    and improve from experience without being explicitly programmed. There are three main \n    types of machine learning: supervised learning, unsupervised learning, and reinforcement \n    learning. Supervised learning uses labeled data to train models, while unsupervised \n    learning finds patterns in unlabeled data. Reinforcement learning learns through \n    interaction with an environment using rewards and penalties.\n    ',
 '\n    Deep Learning and Neural Networks\n    \n    Deep learning is a subset of machine learning based on artificial neural networks. \n    These networks are inspired by the human brain and consist of layers of interconnected \n    nodes. Deep learning has revolutionized fields like computer vision, natural language \n    processing, and speech recognition. Convolutional Neural Networks (CNNs) are particularly \n    eff

In [284]:
for i,doc in enumerate(sample_docs):
    with open(f"data/doc_{i}.txt","w") as f:
        f.write(doc)

### 2. Document Loading

In [285]:
from langchain_community.document_loaders import DirectoryLoader,TextLoader

# Load documents from directory
loader = DirectoryLoader(
    "data",
    glob="*.txt",
    loader_cls=TextLoader,
    loader_kwargs={'encoding': 'utf-8'}
)
documents = loader.load()

print(f"Loaded {len(documents)} documents")
print(f"\nFirst document preview:")
print(documents[0].page_content[:200] + "...")

Loaded 3 documents

First document preview:

    Machine Learning Fundamentals
    
    Machine learning is a subset of artificial intelligence that enables systems to learn 
    and improve from experience without being explicitly programmed. ...


In [286]:
documents

[Document(metadata={'source': 'data/doc_0.txt'}, page_content='\n    Machine Learning Fundamentals\n    \n    Machine learning is a subset of artificial intelligence that enables systems to learn \n    and improve from experience without being explicitly programmed. There are three main \n    types of machine learning: supervised learning, unsupervised learning, and reinforcement \n    learning. Supervised learning uses labeled data to train models, while unsupervised \n    learning finds patterns in unlabeled data. Reinforcement learning learns through \n    interaction with an environment using rewards and penalties.\n    '),
 Document(metadata={'source': 'data/doc_1.txt'}, page_content='\n    Deep Learning and Neural Networks\n    \n    Deep learning is a subset of machine learning based on artificial neural networks. \n    These networks are inspired by the human brain and consist of layers of interconnected \n    nodes. Deep learning has revolutionized fields like computer vision,

### Document Splitting

In [287]:
# Initialize text splitter
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,  # Maximum size of each chunk
    chunk_overlap=50,  # Overlap between chunks to maintain context
    length_function=len,
    separators=[" "]  # Hierarchy of separators
)
chunks=text_splitter.split_documents(documents)

print(f"Created {len(chunks)} chunks from {len(documents)} documents")
print(f"\nChunk example:")
print(f"Content: {chunks[0].page_content[:150]}...")
print(f"Metadata: {chunks[0].metadata}")

Created 5 chunks from 3 documents

Chunk example:
Content: Machine Learning Fundamentals
    
    Machine learning is a subset of artificial intelligence that enables systems to learn 
    and improve from exp...
Metadata: {'source': 'data/doc_0.txt'}


In [288]:
chunks

[Document(metadata={'source': 'data/doc_0.txt'}, page_content='Machine Learning Fundamentals\n    \n    Machine learning is a subset of artificial intelligence that enables systems to learn \n    and improve from experience without being explicitly programmed. There are three main \n    types of machine learning: supervised learning, unsupervised learning, and reinforcement \n    learning. Supervised learning uses labeled data to train models, while unsupervised \n    learning finds patterns in unlabeled data. Reinforcement learning learns through'),
 Document(metadata={'source': 'data/doc_0.txt'}, page_content='data. Reinforcement learning learns through \n    interaction with an environment using rewards and penalties.'),
 Document(metadata={'source': 'data/doc_1.txt'}, page_content='Deep Learning and Neural Networks\n    \n    Deep learning is a subset of machine learning based on artificial neural networks. \n    These networks are inspired by the human brain and consist of layers 

### Embedding Models

In [289]:
from google.colab import userdata
os.environ["GOOGLE_API_KEY"] = userdata.get('GOOGLE_API_KEY')

In [290]:
from langchain_google_genai import GoogleGenerativeAIEmbeddings

sample_text="Machine Learning is fascinating"
embeddings = GoogleGenerativeAIEmbeddings(
    model="gemini-embedding-2"
)
embeddings

GoogleGenerativeAIEmbeddings(client=<google.genai.client.Client object at 0x7a2a8c2f1fa0>, model='gemini-embedding-2', task_type=None, google_api_key=SecretStr('**********'), credentials=None, vertexai=None, project=None, location=None, base_url=None, additional_headers=None, client_args=None, api_version=None, request_options=None, output_dimensionality=None)

In [291]:
vector = embeddings.embed_query(sample_text)
vector

[-0.011709157,
 -0.009887503,
 0.010159971,
 0.01861759,
 -0.0066298475,
 0.007273171,
 -0.017408218,
 -0.002765859,
 0.024916066,
 -0.063009426,
 -0.028653793,
 -0.011862417,
 -0.023967745,
 -0.013139312,
 -0.022249313,
 -0.02207826,
 0.040106896,
 -0.004334457,
 0.0051237755,
 0.009765553,
 0.015837023,
 -0.0054816115,
 -0.0063939444,
 0.021684038,
 -0.009789164,
 0.030231552,
 -0.014571905,
 -0.019517781,
 -0.014175382,
 0.13025884,
 -0.022814536,
 -0.02011869,
 0.0056358525,
 -0.0012096888,
 0.013979278,
 0.008085803,
 0.005192124,
 -0.0035666977,
 -0.013340922,
 -0.0037917832,
 -0.008050911,
 0.010791376,
 0.024550889,
 0.009898351,
 -0.015090927,
 -0.0060495627,
 -0.021839455,
 0.0026165636,
 0.003607197,
 -0.022865586,
 -0.012461826,
 0.014185795,
 0.009203934,
 -0.019273724,
 0.020838061,
 0.0068385187,
 0.010354784,
 -0.006922498,
 0.00694761,
 -6.915422e-05,
 -0.00048952346,
 0.00258065,
 0.017095847,
 0.033524703,
 0.014738813,
 0.0019435248,
 0.029493276,
 -0.036948446,
 0.

### Intilialize the ChromaDB Vector Store And Stores the chunks in Vector Representation

In [292]:
chunks

[Document(metadata={'source': 'data/doc_0.txt'}, page_content='Machine Learning Fundamentals\n    \n    Machine learning is a subset of artificial intelligence that enables systems to learn \n    and improve from experience without being explicitly programmed. There are three main \n    types of machine learning: supervised learning, unsupervised learning, and reinforcement \n    learning. Supervised learning uses labeled data to train models, while unsupervised \n    learning finds patterns in unlabeled data. Reinforcement learning learns through'),
 Document(metadata={'source': 'data/doc_0.txt'}, page_content='data. Reinforcement learning learns through \n    interaction with an environment using rewards and penalties.'),
 Document(metadata={'source': 'data/doc_1.txt'}, page_content='Deep Learning and Neural Networks\n    \n    Deep learning is a subset of machine learning based on artificial neural networks. \n    These networks are inspired by the human brain and consist of layers 

In [293]:
# !pip install chromadb

In [294]:
## Create a Chromdb vector store
persist_directory="./chroma_db"

## Initialize Chromadb with Google Generative AI embeddings
vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    persist_directory=persist_directory,
    collection_name="rag_collection"

)

print(f"Vector store created with {vectorstore._collection.count()} vectors")
print(f"Persisted to: {persist_directory}")

Vector store created with 33 vectors
Persisted to: ./chroma_db


### Test Similarity Search

In [295]:
query = "What are the types of machine learning?"

similar_docs = vectorstore.similarity_search(query,k=3)
similar_docs

[Document(metadata={'source': 'data/doc_0.txt'}, page_content='Machine Learning Fundamentals\n    \n    Machine learning is a subset of artificial intelligence that enables systems to learn \n    and improve from experience without being explicitly programmed. There are three main \n    types of machine learning: supervised learning, unsupervised learning, and reinforcement \n    learning. Supervised learning uses labeled data to train models, while unsupervised \n    learning finds patterns in unlabeled data. Reinforcement learning learns through'),
 Document(metadata={'source': 'data/doc_0.txt'}, page_content='Machine Learning Fundamentals\n    \n    Machine learning is a subset of artificial intelligence that enables systems to learn \n    and improve from experience without being explicitly programmed. There are three main \n    types of machine learning: supervised learning, unsupervised learning, and reinforcement \n    learning. Supervised learning uses labeled data to train mod

In [296]:
query = "what is NLP?"

similar_docs = vectorstore.similarity_search(query,k=3)
similar_docs

[Document(metadata={'source': 'data/doc_2.txt'}, page_content='Natural Language Processing (NLP)\n    \n    NLP is a field of AI that focuses on the interaction between computers and human language. \n    Key tasks in NLP include text classification, named entity recognition, sentiment analysis, \n    machine translation, and question answering. Modern NLP heavily relies on transformer \n    architectures like BERT, GPT, and T5. These models use attention mechanisms to understand \n    context and relationships between words in text.'),
 Document(metadata={'source': 'data/doc_2.txt'}, page_content='Natural Language Processing (NLP)\n    \n    NLP is a field of AI that focuses on the interaction between computers and human language. \n    Key tasks in NLP include text classification, named entity recognition, sentiment analysis, \n    machine translation, and question answering. Modern NLP heavily relies on transformer \n    architectures like BERT, GPT, and T5. These models use attenti

In [297]:
query = "what is Deep Learning?"

similar_docs = vectorstore.similarity_search(query,k=3)
similar_docs

[Document(metadata={'source': 'data/doc_1.txt'}, page_content='Deep Learning and Neural Networks\n    \n    Deep learning is a subset of machine learning based on artificial neural networks. \n    These networks are inspired by the human brain and consist of layers of interconnected \n    nodes. Deep learning has revolutionized fields like computer vision, natural language \n    processing, and speech recognition. Convolutional Neural Networks (CNNs) are particularly \n    effective for image processing, while Recurrent Neural Networks (RNNs) and'),
 Document(metadata={'source': 'data/doc_1.txt'}, page_content='Deep Learning and Neural Networks\n    \n    Deep learning is a subset of machine learning based on artificial neural networks. \n    These networks are inspired by the human brain and consist of layers of interconnected \n    nodes. Deep learning has revolutionized fields like computer vision, natural language \n    processing, and speech recognition. Convolutional Neural Netwo

In [298]:
print(f"Query: {query}")
print(f"\nTop {len(similar_docs)} similar chunks:")
for i, doc in enumerate(similar_docs):
    print(f"\n--- Chunk {i+1} ---")
    print(doc.page_content[:200] + "...")
    print(f"Source: {doc.metadata.get('source', 'Unknown')}")

Query: what is Deep Learning?

Top 3 similar chunks:

--- Chunk 1 ---
Deep Learning and Neural Networks
    
    Deep learning is a subset of machine learning based on artificial neural networks. 
    These networks are inspired by the human brain and consist of layers ...
Source: data/doc_1.txt

--- Chunk 2 ---
Deep Learning and Neural Networks
    
    Deep learning is a subset of machine learning based on artificial neural networks. 
    These networks are inspired by the human brain and consist of layers ...
Source: data/doc_1.txt

--- Chunk 3 ---
Deep Learning and Neural Networks
    
    Deep learning is a subset of machine learning based on artificial neural networks. 
    These networks are inspired by the human brain and consist of layers ...
Source: data/doc_1.txt


### Advanced Similarity Search With Scores

In [299]:
results_scores = vectorstore.similarity_search_with_score("what is NLP?",k=3)
results_scores

[(Document(metadata={'source': 'data/doc_2.txt'}, page_content='Natural Language Processing (NLP)\n    \n    NLP is a field of AI that focuses on the interaction between computers and human language. \n    Key tasks in NLP include text classification, named entity recognition, sentiment analysis, \n    machine translation, and question answering. Modern NLP heavily relies on transformer \n    architectures like BERT, GPT, and T5. These models use attention mechanisms to understand \n    context and relationships between words in text.'),
  0.49015626311302185),
 (Document(metadata={'source': 'data/doc_2.txt'}, page_content='Natural Language Processing (NLP)\n    \n    NLP is a field of AI that focuses on the interaction between computers and human language. \n    Key tasks in NLP include text classification, named entity recognition, sentiment analysis, \n    machine translation, and question answering. Modern NLP heavily relies on transformer \n    architectures like BERT, GPT, and T5

#### Understanding Similarity Scores
The similarity score represents how closely related a document chunk is to your query. The scoring depends on the distance metric used:

ChromaDB default: Uses L2 distance (Euclidean distance)

- Lower scores = MORE similar (closer in vector space)
- Score of 0 = identical vectors
- Typical range: 0 to 2 (but can be higher)


Cosine similarity (if configured):

- Higher scores = MORE similar
- Range: -1 to 1 (1 being identical)

#### Initialize LLM, RAG Chain, Prompt Template , Query the RAG system

In [300]:
# !pip install langchain-groq

In [301]:
"""
from langchain_google_genai import ChatGoogleGenerativeAI

llm = ChatGoogleGenerativeAI(
    model="gemini-3.5-flash"
)
"""

'\nfrom langchain_google_genai import ChatGoogleGenerativeAI\n\nllm = ChatGoogleGenerativeAI(\n    model="gemini-3.5-flash"\n)\n'

In [302]:
from langchain_groq import ChatGroq

llm = ChatGroq(
    model="openai/gpt-oss-120b",
    api_key=userdata.get('GROQ_API_KEY'),
    temperature=0
)

In [303]:
test_response = llm.invoke("What is Large Language Models")
test_response

AIMessage(content='**Large Language Models (LLMs)** are a class of artificial‑intelligence systems that generate or understand natural language by learning statistical patterns from massive amounts of text data.  \n\n---\n\n## 1. Core Idea\n\n- **Statistical prediction:** At their heart, LLMs are probability models. Given a sequence of words (or tokens)\u202f*X* = (x₁, x₂, …, xₙ), the model learns to estimate the probability of the next token *xₙ₊₁*:\n  \\[\n  P(x_{n+1}\\mid x_1,\\dots,x_n)\n  \\]\n- **Training objective:** During training, the model is shown billions of text fragments and is optimized (usually via gradient descent) to maximize the likelihood of the actual next token in each fragment. This forces the network to capture grammar, facts, reasoning patterns, and even some world knowledge.\n\n---\n\n## 2. Architecture\n\n| Component | Typical Implementation | Role |\n|-----------|------------------------|------|\n| **Tokenization** | Byte‑Pair Encoding (BPE), WordPiece, or 

In [304]:
"""
from langchain.chat_models import init_chat_model

# Initialize the Gemini 3.5 Flash model
llm = init_chat_model("google_genai:gemini-3.5-flash")
llm
"""

'\nfrom langchain.chat_models import init_chat_model\n\n# Initialize the Gemini 3.5 Flash model\nllm = init_chat_model("google_genai:gemini-3.5-flash")\nllm\n'

In [305]:
from langchain.chat_models import init_chat_model

llm = init_chat_model(
    "groq:openai/gpt-oss-120b",
    api_key=userdata.get("GROQ_API_KEY")
)

In [306]:
llm.invoke("What is AI")

AIMessage(content='**Artificial Intelligence (AI)** is a branch of computer science that focuses on creating systems capable of performing tasks that normally require human intelligence. These tasks include learning, reasoning, problem‑solving, perception, language understanding, and even creativity.\n\n### Core Ideas\n\n| Concept | What It Means |\n|---------|----------------|\n| **Machine Learning (ML)** | Algorithms that improve their performance automatically through experience (e.g., recognizing images after seeing many examples). |\n| **Deep Learning** | A subset of ML that uses multi‑layered neural networks to model complex patterns (e.g., speech‑to‑text, image generation). |\n| **Natural Language Processing (NLP)** | Techniques for computers to understand, generate, and manipulate human language (e.g., chatbots, translation). |\n| **Computer Vision** | Enabling machines to interpret visual information from cameras or images (e.g., facial recognition, autonomous driving). |\n| *

### Modern RAG Chain

In [307]:
from langchain_classic.chains import create_retrieval_chain
from langchain_core.prompts import ChatPromptTemplate
from langchain_classic.chains.combine_documents import create_stuff_documents_chain

In [308]:
## Convert vector store to retriever
retriever=vectorstore.as_retriever(
    search_kwarg={"k":3} ## Retrieve top 3 relevant chunks
)
retriever

VectorStoreRetriever(tags=['Chroma', 'GoogleGenerativeAIEmbeddings'], vectorstore=<langchain_community.vectorstores.chroma.Chroma object at 0x7a2a96290b90>, search_kwargs={})

In [309]:
## Create a prompt template
from langchain_core.prompts import ChatPromptTemplate
system_prompt="""You are an assistant for question-answering tasks.
Use the following pieces of retrieved context to answer the question.
If you don't know the answer, just say that you don't know.
Use three sentences maximum and keep the answer concise.

Context: {context}"""

prompt = ChatPromptTemplate.from_messages([
    ("system", system_prompt),
    ("human", "{input}")
])

In [310]:
prompt

ChatPromptTemplate(input_variables=['context', 'input'], input_types={}, partial_variables={}, messages=[SystemMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context'], input_types={}, partial_variables={}, template="You are an assistant for question-answering tasks. \nUse the following pieces of retrieved context to answer the question. \nIf you don't know the answer, just say that you don't know. \nUse three sentences maximum and keep the answer concise.\n\nContext: {context}"), additional_kwargs={}), HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['input'], input_types={}, partial_variables={}, template='{input}'), additional_kwargs={})])

##### What is create_stuff_documents_chain?
create_stuff_documents_chain creates a chain that "stuffs" (inserts) all retrieved documents into a single prompt and sends it to the LLM. It's called "stuff" because it literally stuffs all the documents into the context window at once.

In [311]:
### Create a document chain
from langchain_classic.chains.combine_documents import create_stuff_documents_chain
document_chain=create_stuff_documents_chain(llm,prompt)
document_chain

RunnableBinding(bound=RunnableBinding(bound=RunnableAssign(mapper={
  context: RunnableLambda(format_docs)
}), kwargs={}, config={'run_name': 'format_inputs'}, config_factories=[])
| ChatPromptTemplate(input_variables=['context', 'input'], input_types={}, partial_variables={}, messages=[SystemMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context'], input_types={}, partial_variables={}, template="You are an assistant for question-answering tasks. \nUse the following pieces of retrieved context to answer the question. \nIf you don't know the answer, just say that you don't know. \nUse three sentences maximum and keep the answer concise.\n\nContext: {context}"), additional_kwargs={}), HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['input'], input_types={}, partial_variables={}, template='{input}'), additional_kwargs={})])
| ChatGroq(profile={'max_input_tokens': 131072, 'max_output_tokens': 32768, 'image_inputs': False, 'audio_inputs': False, 'video_input

This chain:

- Takes retrieved documents
- "Stuffs" them into the prompt's {context} placeholder
- Sends the complete prompt to the LLM
- Returns the LLM's response

#### What is create_retrieval_chain?
create_retrieval_chain is a function that combines a retriever (which fetches relevant documents) with a document chain (which processes those documents with an LLM) to create a complete RAG pipeline.

In [312]:
### Create The Final RAG Chain
from langchain_classic.chains import create_retrieval_chain
rag_chain=create_retrieval_chain(retriever,document_chain)
rag_chain

RunnableBinding(bound=RunnableAssign(mapper={
  context: RunnableBinding(bound=RunnableLambda(lambda x: x['input'])
           | VectorStoreRetriever(tags=['Chroma', 'GoogleGenerativeAIEmbeddings'], vectorstore=<langchain_community.vectorstores.chroma.Chroma object at 0x7a2a96290b90>, search_kwargs={}), kwargs={}, config={'run_name': 'retrieve_documents'}, config_factories=[])
})
| RunnableAssign(mapper={
    answer: RunnableBinding(bound=RunnableBinding(bound=RunnableAssign(mapper={
              context: RunnableLambda(format_docs)
            }), kwargs={}, config={'run_name': 'format_inputs'}, config_factories=[])
            | ChatPromptTemplate(input_variables=['context', 'input'], input_types={}, partial_variables={}, messages=[SystemMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context'], input_types={}, partial_variables={}, template="You are an assistant for question-answering tasks. \nUse the following pieces of retrieved context to answer the question. \nIf 

In [313]:
response=rag_chain.invoke({"input":"What is Deep LEarning"})

In [314]:
response

{'input': 'What is Deep LEarning',
 'context': [Document(metadata={'source': 'data/doc_1.txt'}, page_content='Deep Learning and Neural Networks\n    \n    Deep learning is a subset of machine learning based on artificial neural networks. \n    These networks are inspired by the human brain and consist of layers of interconnected \n    nodes. Deep learning has revolutionized fields like computer vision, natural language \n    processing, and speech recognition. Convolutional Neural Networks (CNNs) are particularly \n    effective for image processing, while Recurrent Neural Networks (RNNs) and'),
  Document(metadata={'source': 'data/doc_1.txt'}, page_content='Deep Learning and Neural Networks\n    \n    Deep learning is a subset of machine learning based on artificial neural networks. \n    These networks are inspired by the human brain and consist of layers of interconnected \n    nodes. Deep learning has revolutionized fields like computer vision, natural language \n    processing, an

In [315]:
response['answer']

'Deep Learning is a subfield of machine learning that relies on artificial neural networks—computational models inspired by the human brain. These networks are organized in multiple layers of interconnected nodes that learn hierarchical representations from data. Deep learning has driven major advances in areas such as computer vision, natural language processing, and speech recognition.'

In [316]:
# Function to query the modern RAG system
def query_rag_modern(question):
    print(f"Question: {question}")
    print("-" * 50)

    # Using create_retrieval_chain approach
    result = rag_chain.invoke({"input": question})

    print(f"Answer: {result['answer']}")
    print("\nRetrieved Context:")
    for i, doc in enumerate(result['context']):
        print(f"\n--- Source {i+1} ---")
        print(doc.page_content[:200] + "...")

    return result

# Test queries
test_questions = [
    "What are the three types of machine learning?",
    "What is deep learning and how does it relate to neural networks?",
    "What are CNNs best used for?"
]

for question in test_questions:
    result = query_rag_modern(question)
    print("\n" + "="*80 + "\n")

Question: What are the three types of machine learning?
--------------------------------------------------
Answer: The three main types of machine learning are supervised learning, unsupervised learning, and reinforcement learning.

Retrieved Context:

--- Source 1 ---
Machine Learning Fundamentals
    
    Machine learning is a subset of artificial intelligence that enables systems to learn 
    and improve from experience without being explicitly programmed. There...

--- Source 2 ---
Machine Learning Fundamentals
    
    Machine learning is a subset of artificial intelligence that enables systems to learn 
    and improve from experience without being explicitly programmed. There...

--- Source 3 ---
Machine Learning Fundamentals
    
    Machine learning is a subset of artificial intelligence that enables systems to learn 
    and improve from experience without being explicitly programmed. There...

--- Source 4 ---
Machine Learning Fundamentals
    
    Machine learning is a sub

### Create RAG Chain Alternative - Using LCEL (LangChain Expression Language)

In [317]:
# Even more flexible approach using LCEL
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough, RunnableParallel

In [318]:
# Create a custom prompt
custom_prompt = ChatPromptTemplate.from_template("""Use the following context to answer the question.
If you don't know the answer based on the context, say you don't know.
Provide specific details from the context to support your answer.

Context:
{context}

Question: {question}

Answer:""")
custom_prompt

ChatPromptTemplate(input_variables=['context', 'question'], input_types={}, partial_variables={}, messages=[HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context', 'question'], input_types={}, partial_variables={}, template="Use the following context to answer the question. \nIf you don't know the answer based on the context, say you don't know.\nProvide specific details from the context to support your answer.\n\nContext:\n{context}\n\nQuestion: {question}\n\nAnswer:"), additional_kwargs={})])

In [319]:
retriever

VectorStoreRetriever(tags=['Chroma', 'GoogleGenerativeAIEmbeddings'], vectorstore=<langchain_community.vectorstores.chroma.Chroma object at 0x7a2a96290b90>, search_kwargs={})

In [320]:
## Format the output documents for the prompt
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

In [321]:
## Build the chain ussing LCEL

rag_chain_lcel=(
    {
        "context":retriever | format_docs,
        "question": RunnablePassthrough()
     }
    | custom_prompt
    | llm
    | StrOutputParser()
)

rag_chain_lcel

{
  context: VectorStoreRetriever(tags=['Chroma', 'GoogleGenerativeAIEmbeddings'], vectorstore=<langchain_community.vectorstores.chroma.Chroma object at 0x7a2a96290b90>, search_kwargs={})
           | RunnableLambda(format_docs),
  question: RunnablePassthrough()
}
| ChatPromptTemplate(input_variables=['context', 'question'], input_types={}, partial_variables={}, messages=[HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context', 'question'], input_types={}, partial_variables={}, template="Use the following context to answer the question. \nIf you don't know the answer based on the context, say you don't know.\nProvide specific details from the context to support your answer.\n\nContext:\n{context}\n\nQuestion: {question}\n\nAnswer:"), additional_kwargs={})])
| ChatGroq(profile={'max_input_tokens': 131072, 'max_output_tokens': 32768, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': F

In [322]:
response=rag_chain_lcel.invoke("What is Deep Learning")
response

'Deep Learning is a **subset of machine learning** that is built on **artificial neural networks**. These networks are modeled after the human brain and are composed of multiple layers of interconnected nodes. Because of this structure, deep learning has driven major advances in areas such as **computer vision, natural language processing, and speech recognition**. (See the repeated description in the provided context.)'

In [323]:
retriever.invoke("What is Deep Learning")

[Document(metadata={'source': 'data/doc_1.txt'}, page_content='Deep Learning and Neural Networks\n    \n    Deep learning is a subset of machine learning based on artificial neural networks. \n    These networks are inspired by the human brain and consist of layers of interconnected \n    nodes. Deep learning has revolutionized fields like computer vision, natural language \n    processing, and speech recognition. Convolutional Neural Networks (CNNs) are particularly \n    effective for image processing, while Recurrent Neural Networks (RNNs) and'),
 Document(metadata={'source': 'data/doc_1.txt'}, page_content='Deep Learning and Neural Networks\n    \n    Deep learning is a subset of machine learning based on artificial neural networks. \n    These networks are inspired by the human brain and consist of layers of interconnected \n    nodes. Deep learning has revolutionized fields like computer vision, natural language \n    processing, and speech recognition. Convolutional Neural Netwo

In [324]:
# Query using the LCEL approach - Fixed version
def query_rag_lcel(question):
    print(f"Question: {question}")
    print("-" * 50)

    # Method 1: Pass string directly (when using RunnablePassthrough)
    answer = rag_chain_lcel.invoke(question)
    print(f"Answer: {answer}")

    # Get source documents separately if needed
    docs = retriever.invoke(question)
    print("\nSource Documents:")
    for i, doc in enumerate(docs):
        print(f"\n--- Source {i+1} ---")
        print(doc.page_content[:200] + "...")

In [325]:
# Test LCEL chain
print("Testing LCEL Chain:")
query_rag_lcel("What are the key concepts in reinforcement learning?")

Testing LCEL Chain:
Question: What are the key concepts in reinforcement learning?
--------------------------------------------------
Answer: The key concepts in reinforcement learning are **states, actions, rewards, policies, and value functions**. These are explicitly listed in the provided text: “Key concepts in RL include: states, actions, rewards, policies, and value functions.”

Source Documents:

--- Source 1 ---
Reinforcement Learning in Detail

Reinforcement learning (RL) is a type of machine learning where an agent learns to make 
decisions by interacting with an environment. The agent receives rewards or p...

--- Source 2 ---
Reinforcement Learning in Detail

Reinforcement learning (RL) is a type of machine learning where an agent learns to make 
decisions by interacting with an environment. The agent receives rewards or p...

--- Source 3 ---
Reinforcement Learning in Detail

Reinforcement learning (RL) is a type of machine learning where an agent learns to make 
decisions

In [326]:
query_rag_lcel("What is machine learning?")

Question: What is machine learning?
--------------------------------------------------
Answer: Machine learning is defined in the passage as **a subset of artificial intelligence that enables systems to learn and improve from experience without being explicitly programmed**. This description appears repeatedly in the provided text.

Source Documents:

--- Source 1 ---
Machine Learning Fundamentals
    
    Machine learning is a subset of artificial intelligence that enables systems to learn 
    and improve from experience without being explicitly programmed. There...

--- Source 2 ---
Machine Learning Fundamentals
    
    Machine learning is a subset of artificial intelligence that enables systems to learn 
    and improve from experience without being explicitly programmed. There...

--- Source 3 ---
Machine Learning Fundamentals
    
    Machine learning is a subset of artificial intelligence that enables systems to learn 
    and improve from experience without being explicitly pr

In [327]:
query_rag_lcel("What is depe learning?")

Question: What is depe learning?
--------------------------------------------------
Answer: Deep learning is a subset of machine learning that is built on artificial neural networks. These networks are modeled after the human brain and are composed of layers of interconnected nodes. According to the provided context, deep learning “has revolutionized fields like computer vision, natural language processing, and speech recognition.”

Source Documents:

--- Source 1 ---
Deep Learning and Neural Networks
    
    Deep learning is a subset of machine learning based on artificial neural networks. 
    These networks are inspired by the human brain and consist of layers ...

--- Source 2 ---
Deep Learning and Neural Networks
    
    Deep learning is a subset of machine learning based on artificial neural networks. 
    These networks are inspired by the human brain and consist of layers ...

--- Source 3 ---
Deep Learning and Neural Networks
    
    Deep learning is a subset of machine lea

### Add New Documents To Existing Vector Store

In [328]:
vectorstore

In [329]:
# Add new documents to the existing vector store
new_document = """
Reinforcement Learning in Detail

Reinforcement learning (RL) is a type of machine learning where an agent learns to make
decisions by interacting with an environment. The agent receives rewards or penalties
based on its actions and learns to maximize cumulative reward over time. Key concepts
in RL include: states, actions, rewards, policies, and value functions. Popular RL
algorithms include Q-learning, Deep Q-Networks (DQN), Policy Gradient methods, and
Actor-Critic methods. RL has been successfully applied to game playing (like AlphaGo),
robotics, and autonomous systems.
"""

In [330]:
new_document

'\nReinforcement Learning in Detail\n\nReinforcement learning (RL) is a type of machine learning where an agent learns to make \ndecisions by interacting with an environment. The agent receives rewards or penalties \nbased on its actions and learns to maximize cumulative reward over time. Key concepts \nin RL include: states, actions, rewards, policies, and value functions. Popular RL \nalgorithms include Q-learning, Deep Q-Networks (DQN), Policy Gradient methods, and \nActor-Critic methods. RL has been successfully applied to game playing (like AlphaGo), \nrobotics, and autonomous systems.\n'

In [331]:
chunks

[Document(metadata={'source': 'data/doc_0.txt'}, page_content='Machine Learning Fundamentals\n    \n    Machine learning is a subset of artificial intelligence that enables systems to learn \n    and improve from experience without being explicitly programmed. There are three main \n    types of machine learning: supervised learning, unsupervised learning, and reinforcement \n    learning. Supervised learning uses labeled data to train models, while unsupervised \n    learning finds patterns in unlabeled data. Reinforcement learning learns through'),
 Document(metadata={'source': 'data/doc_0.txt'}, page_content='data. Reinforcement learning learns through \n    interaction with an environment using rewards and penalties.'),
 Document(metadata={'source': 'data/doc_1.txt'}, page_content='Deep Learning and Neural Networks\n    \n    Deep learning is a subset of machine learning based on artificial neural networks. \n    These networks are inspired by the human brain and consist of layers 

In [332]:
new_doc=Document(
    page_content=new_document,
    metadata={"source": "manual_addition", "topic": "reinforcement_learning"}
)

In [333]:
new_doc

Document(metadata={'source': 'manual_addition', 'topic': 'reinforcement_learning'}, page_content='\nReinforcement Learning in Detail\n\nReinforcement learning (RL) is a type of machine learning where an agent learns to make \ndecisions by interacting with an environment. The agent receives rewards or penalties \nbased on its actions and learns to maximize cumulative reward over time. Key concepts \nin RL include: states, actions, rewards, policies, and value functions. Popular RL \nalgorithms include Q-learning, Deep Q-Networks (DQN), Policy Gradient methods, and \nActor-Critic methods. RL has been successfully applied to game playing (like AlphaGo), \nrobotics, and autonomous systems.\n')

In [334]:
## split the documents
new_chunks=text_splitter.split_documents([new_doc])
new_chunks

[Document(metadata={'source': 'manual_addition', 'topic': 'reinforcement_learning'}, page_content='Reinforcement Learning in Detail\n\nReinforcement learning (RL) is a type of machine learning where an agent learns to make \ndecisions by interacting with an environment. The agent receives rewards or penalties \nbased on its actions and learns to maximize cumulative reward over time. Key concepts \nin RL include: states, actions, rewards, policies, and value functions. Popular RL \nalgorithms include Q-learning, Deep Q-Networks (DQN), Policy Gradient methods, and \nActor-Critic methods. RL has been'),
 Document(metadata={'source': 'manual_addition', 'topic': 'reinforcement_learning'}, page_content='methods, and \nActor-Critic methods. RL has been successfully applied to game playing (like AlphaGo), \nrobotics, and autonomous systems.')]

In [335]:
### Add new documents to vectorstore
vectorstore.add_documents(new_chunks)



['e7bdb374-f96b-4dc4-b12a-2fcbd66e0c08',
 'a9ad0f2a-a435-4cc9-8ef7-3cd57359dfbf']

In [336]:
print(f"Added {len(new_chunks)} new chunks to the vector store")
print(f"Total vectors now: {vectorstore._collection.count()}")

Added 2 new chunks to the vector store
Total vectors now: 35


In [337]:
## query with the updated vector
new_question="What are the keys concepts in reinforcement learning"
result=query_rag_lcel(new_question)
result

Question: What are the keys concepts in reinforcement learning
--------------------------------------------------
Answer: The key concepts in reinforcement learning are **states, actions, rewards, policies, and value functions**. The context explicitly lists them: “Key concepts in RL include: states, actions, rewards, policies, and value functions.”

Source Documents:

--- Source 1 ---
Reinforcement Learning in Detail

Reinforcement learning (RL) is a type of machine learning where an agent learns to make 
decisions by interacting with an environment. The agent receives rewards or p...

--- Source 2 ---
Reinforcement Learning in Detail

Reinforcement learning (RL) is a type of machine learning where an agent learns to make 
decisions by interacting with an environment. The agent receives rewards or p...

--- Source 3 ---
Reinforcement Learning in Detail

Reinforcement learning (RL) is a type of machine learning where an agent learns to make 
decisions by interacting with an environment

### Advanced Rag Techniques- Conversational Memory
Understanding Conversational Memory in RAG
Conversational memory enables RAG systems to maintain context across multiple interactions. This is crucial for:

Follow-up questions that reference previous answers
Pronoun resolution (e.g., "it", "they", "that")
Context-dependent queries that build on prior discussion
Natural dialogue flow where users don't repeat context

Key Challenge:
Traditional RAG retrieves documents based only on the current query, missing important context from the conversation. For example:

User: "Tell me about Python"
Bot: explains Python programming language
User: "What are its main libraries?" ← "its" refers to Python, but retriever doesn't know this

Solution:
The modern approach uses a two-step process:

Query Reformulation: Transform context-dependent questions into standalone queries
Context-Aware Retrieval: Use the reformulated query to fetch relevant documents

- create_history_aware_retriever: Makes the retriever understand conversation context
- MessagesPlaceholder: Placeholder for chat history in prompts
- HumanMessage/AIMessage: Structured message types for conversation history

In [338]:
from langchain_classic.chains import create_history_aware_retriever
from langchain_core.prompts import MessagesPlaceholder
from langchain_core.messages import HumanMessage, AIMessage

In [339]:
## create a prompt that includes the chat history
contextualize_q_system_prompt = """Given a chat history and the latest user question \
which might reference context in the chat history, formulate a standalone question \
which can be understood without the chat history. Do NOT answer the question, \
just reformulate it if needed. If the reformulated question would be empty or \
if no reformulation is necessary, return the latest user question as is. \
The output must always be a non-empty question string."""

contextualize_q_prompt = ChatPromptTemplate.from_messages([
    ("system", contextualize_q_system_prompt),
    MessagesPlaceholder("chat_history"),
    ("human", "{input}"),
])

In [340]:
## create history aware retriever
history_aware_retriever = create_history_aware_retriever(
    llm, retriever, contextualize_q_prompt
)
history_aware_retriever

RunnableBinding(bound=RunnableBranch(branches=[(RunnableLambda(lambda x: not x.get('chat_history', False)), RunnableLambda(lambda x: x['input'])
| VectorStoreRetriever(tags=['Chroma', 'GoogleGenerativeAIEmbeddings'], vectorstore=<langchain_community.vectorstores.chroma.Chroma object at 0x7a2a96290b90>, search_kwargs={}))], default=ChatPromptTemplate(input_variables=['chat_history', 'input'], input_types={'chat_history': list[typing.Annotated[typing.Union[typing.Annotated[langchain_core.messages.ai.AIMessage, Tag(tag='ai')], typing.Annotated[langchain_core.messages.human.HumanMessage, Tag(tag='human')], typing.Annotated[langchain_core.messages.chat.ChatMessage, Tag(tag='chat')], typing.Annotated[langchain_core.messages.system.SystemMessage, Tag(tag='system')], typing.Annotated[langchain_core.messages.function.FunctionMessage, Tag(tag='function')], typing.Annotated[langchain_core.messages.tool.ToolMessage, Tag(tag='tool')], typing.Annotated[langchain_core.messages.ai.AIMessageChunk, Tag(

In [341]:
# Create a new document chain with history
qa_system_prompt = """You are an assistant for question-answering tasks.
Use the following pieces of retrieved context to answer the question.
If you don't know the answer, just say that you don't know.
Use three sentences maximum and keep the answer concise.

Context: {context}"""

qa_prompt = ChatPromptTemplate.from_messages([
    ("system", qa_system_prompt),
    MessagesPlaceholder("chat_history"),
    ("human", "{input}"),
])

question_answer_chain = create_stuff_documents_chain(llm, qa_prompt)

# Create conversational RAG chain
conversational_rag_chain = create_retrieval_chain(
    history_aware_retriever,
    question_answer_chain
)
print("Conversational RAG chain created!")

Conversational RAG chain created!


In [342]:
chat_history=[]
# First question
result1 = conversational_rag_chain.invoke({
    "chat_history": chat_history,
    "input": "What is machine learning?"
})
print(f"Q: What is machine learning?")
print(f"A: {result1['answer']}")

Q: What is machine learning?
A: Machine learning is a subset of artificial intelligence that enables systems to learn from data and improve their performance through experience, without being explicitly programmed. It involves algorithms that automatically identify patterns and make predictions or decisions.


In [343]:
chat_history.extend([
    HumanMessage(content="What is machine learning"),
    AIMessage(content=result1['answer'])
])

In [344]:
chat_history

[HumanMessage(content='What is machine learning', additional_kwargs={}, response_metadata={}),
 AIMessage(content='Machine learning is a subset of artificial intelligence that enables systems to learn from data and improve their performance through experience, without being explicitly programmed. It involves algorithms that automatically identify patterns and make predictions or decisions.', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[])]

In [345]:
## Follow up question

# Re-create history aware retriever with the updated contextualize_q_prompt
# Ensure `llm`, `retriever`, `contextualize_q_prompt`, `qa_system_prompt`, `qa_prompt` are in scope

from langchain_core.runnables import RunnableBranch, RunnableLambda, RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

# Custom question generation with a fallback to handle empty LLM responses
question_generator_chain = contextualize_q_prompt | llm | StrOutputParser()

def get_standalone_question(input_dict):
    reformulated_q = question_generator_chain.invoke(input_dict)

    # Ensure reformulated_q is a string and handle potential None or non-string outputs
    if not isinstance(reformulated_q, str):
        reformulated_q_str = "" # Treat non-string or None as empty string
    else:
        reformulated_q_str = reformulated_q.strip() # Strip whitespace

    # If the cleaned reformulated question is empty, use the original input
    if not reformulated_q_str:
        final_question = input_dict["input"]
    else:
        final_question = reformulated_q_str

    # Final safeguard: ensure the question is never truly empty
    if not final_question:
        # Fallback to a generic but valid question if all else (unexpectedly) fails
        return "generic search query"

    return final_question

# Create the history-aware retriever using LCEL, incorporating the fallback
history_aware_retriever = RunnableBranch(
    (
        lambda x: not x.get("chat_history", False), # Condition: no chat_history
        RunnableLambda(lambda x: retriever.invoke(x["input"])) # If no history, retrieve with original input
    ),
    # If chat_history exists, generate a standalone question with fallback, then retrieve
    RunnableLambda(get_standalone_question) | retriever
)

qa_system_prompt = """You are an assistant for question-answering tasks.
Use the following pieces of retrieved context to answer the question.
If you don't know the answer, just say that you don't know.
Use three sentences maximum and keep the answer concise.

Context: {context}"""

qa_prompt = ChatPromptTemplate.from_messages([
    ("system", qa_system_prompt),
    MessagesPlaceholder("chat_history"),
    ("human", "{input}"),
])

question_answer_chain = create_stuff_documents_chain(llm, qa_prompt)

conversational_rag_chain = create_retrieval_chain(
    history_aware_retriever,
    question_answer_chain
)

# Follow-up question
result2 = conversational_rag_chain.invoke({
    "chat_history": chat_history,
    "input": "What are its main types?"  # Refers to ML from previous question
})
result2

{'chat_history': [HumanMessage(content='What is machine learning', additional_kwargs={}, response_metadata={}),
  AIMessage(content='Machine learning is a subset of artificial intelligence that enables systems to learn from data and improve their performance through experience, without being explicitly programmed. It involves algorithms that automatically identify patterns and make predictions or decisions.', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[])],
 'input': 'What are its main types?',
 'context': [Document(metadata={'source': 'data/doc_0.txt'}, page_content='Machine Learning Fundamentals\n    \n    Machine learning is a subset of artificial intelligence that enables systems to learn \n    and improve from experience without being explicitly programmed. There are three main \n    types of machine learning: supervised learning, unsupervised learning, and reinforcement \n    learning. Supervised learning uses labeled data to train models, while

In [346]:
result2['answer']

'The three main types of machine learning are **supervised learning**, which trains models on labeled data; **unsupervised learning**, which discovers patterns in unlabeled data; and **reinforcement learning**, which learns optimal actions through trial‑and‑error feedback from the environment.'